In [46]:
import pickle
import os
import copy
import matplotlib.pyplot as plt
import numpy as np
import paper_style  # apply the style automatically
import gc
import psutil
import yaml
import sys
import torch
import io
from matplotlib.lines import Line2D
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns




# Input Config

In [47]:
df = pd.read_csv('/home/evrond/rl_for_curobo_analysis/projects_root/experiments/armsResults_date1409_rows960_rowTypeTrial.csv')
df

,sim_id,alg,task_type,task_level,task_seed,col_chance,task_val
0,2025-09-08_03:30:27_R_ur5e_N4_AO_Tbin_s5_l2,"O(400, 1)",bin,2,5,0.000,6.000000
1,2025-09-08_04:01:24_R_ur5e_N4_AO_Tbin_s3_l4,"O(400, 1)",bin,4,3,0.022,7.000000
2,2025-09-08_03:27:28_R_ur5e_N4_AO_Tbin_s4_l2,"O(400, 1)",bin,2,4,0.000,6.000000
3,2025-09-08_04:23:09_R_ur5e_N4_AO_Tbin_s4_l5,"O(400, 1)",bin,5,4,0.000,8.000000
4,2025-09-08_04:04:41_R_ur5e_N4_AO_Tbin_s4_l4,"O(400, 1)",bin,4,4,0.000,6.000000
...,...,...,...,...,...,...,...
955,2025-09-08_22:54:49_R_ur5e_N4_ACC_Tfollow_s5_l2,CC(500),follow,2,5,0.000,0.304553
956,2025-09-08_22:41:53_R_ur5e_N4_ACC_Tfollow_s2_l2,CC(500),follow,2,2,0.000,0.297082
957,2025-09-08_22:22:11_R_ur5e_N4_ACC_Tfollow_s3_l1,CC(500),follow,1,3,0.000,0.310030
958,2025-09-08_23:40:16_R_ur5e_N4_ACC_Tfollow_s3_l4,CC(500),follow,4,3,0.278,0.307996


In [56]:
def normalize(df, baseline_alg='O(400, 5)', normalization_type='diff_from_baseline'):
    """
    normalization_type: "diff_from_baseline" or "ratio_from_max"
        diff_from_baseline: normalized_task_val = task_val - baseline_alg's task_val at the same (task_type, task_seed, task_level)
        ratio_from_max: normalized_task_val = task_val / max task_val over all algs at the same(task_type, task_seed, task_level)
    
    """
    
    df_work = df.copy()
    df_work['normalized_task_val'] = 0.0
    df_work['normalized_col_chance'] = 0.0
    
    # Get all unique combinations of task_type, task_seed, task_level
    unique_combinations = df_work[['task_type', 'task_seed', 'task_level']].drop_duplicates() # df with all unique combinations of task_type, task_seed, task_level, each appears only once
    
    for idx, row in unique_combinations.iterrows():
        # print(idx, row)
        task_type = row['task_type']
        task_seed = row['task_seed'] 
        task_level = row['task_level']
        
        # Find the baseline_alg rows with matching task_type, task_seed, task_level
        baseline_alg_mask = (
            (df_work['alg'] == baseline_alg) & 
            (df_work['task_type'] == task_type) & 
            (df_work['task_seed'] == task_seed) & 
            (df_work['task_level'] == task_level)
        )
        
        baseline_alg_rows = df_work[baseline_alg_mask]
        
        # Sanity check: verify there is exactly one baseline_alg row
        if len(baseline_alg_rows) == 0:
            raise ValueError(f"No {baseline_alg} row found for task_type={task_type}, "
                           f"task_seed={task_seed}, task_level={task_level}")
        elif len(baseline_alg_rows) > 1:
            raise ValueError(f"Multiple {baseline_alg} rows found for task_type={task_type}, "
                           f"task_seed={task_seed}, task_level={task_level}. Found {len(baseline_alg_rows)} rows.")
        
        # Get the baseline_alg task_val
        baseline_alg_task_val = baseline_alg_rows['task_val'].iloc[0]
        baseline_alg_col_chance = baseline_alg_rows['col_chance'].iloc[0]
        
        # Compute deviation for all rows with this combination
        combination_mask = (
            (df_work['task_type'] == task_type) & 
            (df_work['task_seed'] == task_seed) & 
            (df_work['task_level'] == task_level)
        )
        
        if normalization_type == 'diff_from_baseline':
            # normalize task_val - diff from baseline
            df_work.loc[combination_mask, 'normalized_task_val'] = (
                    df_work.loc[combination_mask, 'task_val'] - baseline_alg_task_val
                )

            # normalize col_chance - diff from baseline
            df_work.loc[combination_mask, 'normalized_col_chance'] = (
                    df_work.loc[combination_mask, 'col_chance'] - baseline_alg_col_chance
                )
        elif normalization_type == 'ratio_from_max':

            # normalize task_val - ratio from max
            max_task_val = df_work.loc[combination_mask, 'task_val'].max()
            if max_task_val == 0:
                normalized_ratio = 1 # avoid division by zero
            else:  
                normalized_ratio = df_work.loc[combination_mask, 'task_val'] / max_task_val  
            df_work.loc[combination_mask, 'normalized_task_val'] = normalized_ratio
            
            # normalize col_chance - ratio from max
            max_col_chance = df_work.loc[combination_mask, 'col_chance'].max()
            if max_col_chance == 0:
                normalized_col_chance = 1 # avoid division by zero
            else:  
                normalized_col_chance = df_work.loc[combination_mask, 'col_chance'] / max_col_chance
            df_work.loc[combination_mask, 'normalized_col_chance'] = normalized_col_chance
            
    
    return df_work

In [65]:
df_normalized_diff_from_O400_5 = normalize(df)
df_normalized_ratio_from_max = normalize(df, normalization_type='ratio_from_max')




In [67]:
df_normalized_diff_from_O400_5

,sim_id,alg,task_type,task_level,task_seed,col_chance,task_val,normalized_task_val,normalized_col_chance
0,2025-09-08_03:30:27_R_ur5e_N4_AO_Tbin_s5_l2,"O(400, 1)",bin,2,5,0.000,6.000000,1.000000,0.000
1,2025-09-08_04:01:24_R_ur5e_N4_AO_Tbin_s3_l4,"O(400, 1)",bin,4,3,0.022,7.000000,0.000000,0.022
2,2025-09-08_03:27:28_R_ur5e_N4_AO_Tbin_s4_l2,"O(400, 1)",bin,2,4,0.000,6.000000,-2.000000,0.000
3,2025-09-08_04:23:09_R_ur5e_N4_AO_Tbin_s4_l5,"O(400, 1)",bin,5,4,0.000,8.000000,-1.000000,0.000
4,2025-09-08_04:04:41_R_ur5e_N4_AO_Tbin_s4_l4,"O(400, 1)",bin,4,4,0.000,6.000000,-1.000000,0.000
...,...,...,...,...,...,...,...,...,...
955,2025-09-08_22:54:49_R_ur5e_N4_ACC_Tfollow_s5_l2,CC(500),follow,2,5,0.000,0.304553,0.208860,0.000
956,2025-09-08_22:41:53_R_ur5e_N4_ACC_Tfollow_s2_l2,CC(500),follow,2,2,0.000,0.297082,0.181914,0.000
957,2025-09-08_22:22:11_R_ur5e_N4_ACC_Tfollow_s3_l1,CC(500),follow,1,3,0.000,0.310030,0.216714,0.000
958,2025-09-08_23:40:16_R_ur5e_N4_ACC_Tfollow_s3_l4,CC(500),follow,4,3,0.278,0.307996,0.202744,0.278


In [66]:
df_normalized_diff_from_O400_5_gb_task = df_normalized_diff_from_O400_5.groupby(['task_type'])
df_normalized_diff_from_O400_5_gb_task_and_level = df_normalized_diff_from_O400_5.groupby(['task_type', 'task_level'])
